# DeepGuard — AV-Deepfake1M++ Drive Pilot

Modern external benchmark alternative while DF40 Google Drive quota is unavailable. AV-Deepfake1M++ is the dataset used in the 2025 1M-Deepfakes Detection Challenge. It contains about 2M videos and the official Hugging Face repository is gated by registration/EULA. This notebook downloads metadata first and then lets the user choose a manageable subset.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from pathlib import Path
import shutil, subprocess, sys, os
ROOT=Path('/content/drive/MyDrive/DeepGuard')
DATA=ROOT/'datasets/AV-Deepfake1M-PlusPlus'
DATA.mkdir(parents=True,exist_ok=True)
print('Target:',DATA)
print('Free Drive GiB:',round(shutil.disk_usage('/content/drive').free/1024**3,1))


## 1. Accept the official terms first

Open the official dataset page and complete the registration/EULA. The dataset is gated and requires contact information/approval.

Official dataset: https://huggingface.co/datasets/ControlNet/AV-Deepfake1M-PlusPlus

Registration: https://deepfakes1m.github.io/2025/registration


In [ ]:
!pip -q install -U huggingface_hub
from huggingface_hub import login
print('Paste a Hugging Face access token with access to AV-Deepfake1M++ when prompted.')
login()


In [ ]:
from huggingface_hub import HfApi
api=HfApi()
repo='ControlNet/AV-Deepfake1M-PlusPlus'
info=api.dataset_info(repo)
print('Dataset:',info.id)
print('Gated access confirmed if this call succeeds.')


In [ ]:
# Download only lightweight metadata/file lists first. No multi-terabyte video data yet.
from huggingface_hub import snapshot_download
meta=DATA/'metadata'
meta.mkdir(parents=True,exist_ok=True)
snapshot_download(repo_id=repo,repo_type='dataset',local_dir=str(meta),allow_patterns=['README.md','train_metadata.json','val_metadata.json','testA_files.txt','testB_files.txt'])
print('Metadata downloaded to:',meta)


In [ ]:
# Optional: download one complete subset. TestB is the smallest official test subset (~50k videos).
# The full collection is ~1.46 TB, so do NOT run the full snapshot blindly.
SUBSET='testB'
print('Selected subset:',SUBSET)
print('The next cell downloads all testB zip parts; check Drive space first.')


In [ ]:
free=shutil.disk_usage('/content/drive').free/1024**3
if free < 160:
    raise RuntimeError(f'Only {free:.1f} GiB free. Refusing to start a large subset download.')
print('Enough Drive space for a pilot. Starting official testB download...')
from huggingface_hub import snapshot_download
snapshot_download(repo_id=repo,repo_type='dataset',local_dir=str(DATA/'testB'),allow_patterns=['testB/*','testB_files.txt'])
print('Download finished.')


## 2. Next DeepGuard step

After testB is available, we can add an adapter that samples the videos, runs Xception/FTCN, stores video-level scores and feeds them into DeepGuard-LR. TestA/TestB are challenge evaluation sets, so they must not be used for LR calibration.
